# Python for AI — Class 12
### Errors (exceptions) and files

Until now, one bad value could stop your whole program. Today you will learn two things:

1. **Handle errors.** Catch an error, and decide what to do next. You can also make your own errors.
2. **Use files.** Save data to a file and read it back: text files, CSV files and JSON files.

When a program stops, its variables are lost. Files are not lost. Every AI project loads data from a file at the start and saves results to a file at the end.

**How to use this notebook:** run each cell with the play button, or press `Shift + Enter`.

Some cells say **BREAKS ON PURPOSE**. They will show a red error. This is planned.
Some cells say **WRONG ON PURPOSE**. They run with no error, but the answer is wrong. These are the dangerous ones.
Do not fix them before class. We will learn from them together.

# 1. Why do we need this? One error stops everything

In Class 2 you learned to read an error message. Now see the real problem: one bad value, and the program stops. The lines after it never run.

In [1]:
# BREAKS ON PURPOSE - one bad value kills the whole loop
prices = ["1200", "850", "free", "400"]

total = 0
for p in prices:
    total += int(p)          # int("free") -> ValueError
    print("running total:", total)

print("Total:", total)       # never reached

running total: 1200
running total: 2050


ValueError: invalid literal for int() with base 10: 'free'

**ValueError: invalid literal for int() with base 10: 'free'.**

- The first two prices were fine.
- The program stopped at the third price, `"free"`.
- The fourth price was never checked.

Imagine a file with 10,000 rows. One bad row would stop all the work.

# 2. `try` and `except`: catch the error and continue

Put the risky line inside `try`. If it gives an error, Python does not stop. It jumps to `except` and runs that code instead:

```
try:
    risky line          ← if this gives an error...
except ValueError:
    do this instead     ← ...Python runs this, then continues
```

In [2]:
prices = ["1200", "850", "free", "400"]

total = 0
for p in prices:
    try:
        total += int(p)
    except ValueError:
        print(f"Skipping bad price: {p!r}")

print("Total:", total)

Skipping bad price: 'free'
Total: 2450


Same data. But now the bad price is printed and skipped. The other three prices are added.

This is the main idea: **you** decide what happens when there is an error. Python does not stop the program.

## Common errors in Python

You will see these errors many times. The name of the error tells you the problem.

| Error name | When it happens | Example |
|------------|-----------------|---------|
| `ValueError` | the type is right, but the value is wrong | `int("abc")` |
| `TypeError` | the type is wrong | `"5" + 5` |
| `ZeroDivisionError` | dividing by zero | `10 / 0` |
| `IndexError` | the list position does not exist | `[1, 2][5]` |
| `KeyError` | the dictionary key does not exist | `{"a": 1}["b"]` |
| `FileNotFoundError` | the file does not exist | `open("abc.txt")` |
| `AttributeError` | the object does not have this attribute or method | you saw this in Class 11 |

In [1]:
# Trigger each one on purpose and catch it - the message tells you exactly what broke
tests = [
    lambda: int("abc"),
    lambda: "5" + 5,
    lambda: 10 / 0,
    lambda: [1, 2][5],
    lambda: {"a": 1}["b"],
    lambda: open("does_not_exist.txt"),
]

for t in tests:
    try:
        t()
    except Exception as e:                      # 'as e' gives you the error object
        print(f"{type(e).__name__:<18} -> {e}")

ValueError         -> invalid literal for int() with base 10: 'abc'
TypeError          -> can only concatenate str (not "int") to str
ZeroDivisionError  -> division by zero
IndexError         -> list index out of range
KeyError           -> 'b'
FileNotFoundError  -> [Errno 2] No such file or directory: 'does_not_exist.txt'


## Many `except` blocks, plus `else` and `finally`

One `try` can have many `except` blocks. Python uses the first one that matches the error.

There are two more parts. You can use them if you need them:

| Block | When does it run? |
|-------|-------------------|
| `try` | always. Put the risky code here |
| `except X` | only if error `X` happens |
| `else` | only if there was **no** error |
| `finally` | **always**, with or without an error |

In [4]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  Can't divide by zero")
    except TypeError:
        print("  Both values must be numbers")
    else:
        print("  Result:", result)          # only on success
    finally:
        print("  -- divide() finished --")  # always

print("divide(10, 2)");   divide(10, 2)
print("divide(10, 0)");   divide(10, 0)
print("divide(10, 'x')"); divide(10, "x")

divide(10, 2)
  Result: 5.0
  -- divide() finished --
divide(10, 0)
  Can't divide by zero
  -- divide() finished --
divide(10, 'x')
  Both values must be numbers
  -- divide() finished --


Look: `finally` printed every time.

So we use `finally` for work that must happen **every time**. For example: closing a file, or closing a database connection.

## Be careful: `except:` with no error name hides your mistakes

In [ ]:
# WRONG ON PURPOSE - catches everything, including your own typo
def average(marks):
    try:
        return sum(marks) / len(mark)      # typo: 'mark' instead of 'marks'
    except:
        return 0                           # "no marks" - or is it?

print(average([80, 90, 70]))               # prints 0. The real answer is 80.

0


It printed `0` and showed no error. But the right answer is 80.

What happened?
- The code has a spelling mistake: `mark` instead of `marks`. This gives a `NameError`.
- `except:` caught **every** error, including this one.
- So the mistake was hidden, and we got a wrong answer.

**Rule:** always write the error name you expect, like `except ZeroDivisionError:`. Never write only `except:`.

# 3. Make your own errors with `raise`

Sometimes Python has no problem, but the data breaks **your** rules. For example: a price below zero, or an age of 300.

Use `raise` to stop the program and show a clear message:

In [3]:
def set_age(age):
    if not 0 <= age <= 120:
        raise ValueError(f"Age must be 0-120, got {age}")
    return age

print(set_age(25))

try:
    set_age(300)
except ValueError as e:
    print("Rejected:", e)

25
Rejected: Age must be 0-120, got 300


## Your own error type

You can make your own error type. It is just a class that inherits from `Exception`. (This is the inheritance you learned in Class 11.)

Then other code can catch **your** error by its name.

In [4]:
class InsufficientFundsError(Exception):
    pass


class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(f"{self.owner} has Rs {self.balance}, tried Rs {amount}")
        self.balance -= amount


acc = Account("Ali", 5000)
try:
    acc.withdraw(8000)
except InsufficientFundsError as e:
    print("Transaction declined:", e)

Transaction declined: Ali has Rs 5000, tried Rs 8000


In Class 11, `withdraw` only printed "Refused".

- A `print` only shows a message. The rest of the code does not know that anything went wrong.
- An error with `raise` **stops** the code. The code must handle it.
- The name `InsufficientFundsError` tells us exactly what the problem is: not enough money.

# 4. Files: write and read text

When a program stops, its variables are lost. A file stays on the computer.

`open(filename, mode)` opens a file. The **mode** tells Python what you want to do:

| Mode | Meaning |
|------|---------|
| `"r"` | read the file (this is the default). Error if the file does not exist |
| `"w"` | write. Makes a new file, or **deletes everything** in the old file first |
| `"a"` | append. Adds new text at the end of the file |

Always write `with open(...) as f:`. Python will close the file for you when the block ends, even if there is an error.

In [ ]:
with open("notes.txt", "w") as f:
    f.write("Class 12: exceptions\n")
    f.write("Class 12: files\n")

with open("notes.txt", "a") as f:
    f.write("Homework: practice set\n")

with open("notes.txt") as f:
    print(f.read())

In [ ]:
# Reading line by line - the normal way for big files
with open("notes.txt") as f:
    for number, line in enumerate(f, start=1):
        print(number, line.strip())        # strip() removes the trailing \n

In [ ]:
# WRONG ON PURPOSE - "w" wipes the file before writing
with open("notes.txt", "w") as f:
    f.write("One more note\n")

with open("notes.txt") as f:
    print(f.read())                        # the first three lines are gone

There was no error, but three lines were deleted.

`"w"` means "start a new empty file". To add text at the end, use `"a"`. Many real files have been lost because of this mistake.

In [ ]:
# Missing file = FileNotFoundError - catch it and give a useful message
try:
    with open("marks_2023.txt") as f:
        print(f.read())
except FileNotFoundError:
    print("marks_2023.txt not found - check the file name and folder")

# 5. CSV files: data in rows and columns

A CSV file is like an Excel sheet saved as plain text. CSV means "comma-separated values": a comma goes between the values.

Most datasets come as CSV files. Python has a built-in `csv` module to read and write them.

- `DictWriter` writes dictionaries as rows.
- `DictReader` reads each row back as a dictionary.

In [ ]:
import csv

students = [
    {"name": "Ali",    "city": "Lahore",    "marks": 82},
    {"name": "Sara",   "city": "Karachi",   "marks": 91},
    {"name": "Hassan", "city": "Islamabad", "marks": 67},
]

with open("students.csv", "w", newline="") as f:     # newline="" avoids blank lines on Windows
    writer = csv.DictWriter(f, fieldnames=["name", "city", "marks"])
    writer.writeheader()
    writer.writerows(students)

with open("students.csv") as f:
    print(f.read())                                   # this is what the file really looks like

In [ ]:
# WRONG ON PURPOSE - everything read from a CSV is a string
with open("students.csv") as f:
    rows = list(csv.DictReader(f))

print(rows[0])
print("Top marks:", max(r["marks"] for r in rows))   # compares text: "91" > "82" > "67" ... looks fine?

In [ ]:
# ...until a mark has a different number of digits
rows.append({"name": "Zara", "city": "Multan", "marks": "100"})
print("Top marks:", max(r["marks"] for r in rows))   # says 91! "9" > "1" as text

CSV files do not save types. Every value comes back as **text** (a string).

- Python compares text one letter at a time.
- `"100"` starts with `"1"`. `"91"` starts with `"9"`.
- `"9"` is bigger than `"1"`, so `"91"` wins. This is wrong.

**Fix:** change the text to a number when you read it: `int(r["marks"])`. A real CSV file can have bad rows, so put this inside a `try`.

In [ ]:
with open("students.csv") as f:
    rows = list(csv.DictReader(f))

for r in rows:
    r["marks"] = int(r["marks"])

best = max(rows, key=lambda r: r["marks"])           # Class 7's key= trick
print(f"Top student: {best['name']} ({best['marks']})")

# 6. JSON files: data inside data

A CSV file has only simple rows. A **JSON** file can have lists and dictionaries inside other dictionaries, just like Python.

JSON is used by websites and APIs. In Month 2, every AI (LLM) API will send you JSON.

| Function | What it does | Data moves from → to |
|----------|--------------|----------------------|
| `json.dump(obj, f)` | **save** a dict/list into a file | Python dict → `.json` file |
| `json.load(f)` | **read** a file into a dict/list | `.json` file → Python dict |
| `json.dumps(obj)` | change a dict into JSON **text** (no file) | Python dict → string |
| `json.loads(text)` | change JSON **text** into a dict | string → Python dict |

The `s` at the end means *string*.
- Working with a **file**? Use `dump` and `load`.
- Working with **text in a variable**, like a reply from an API? Use `dumps` and `loads`.

In [ ]:
import json

config = {
    "model": "gpt-4o-mini",
    "temperature": 0.2,
    "max_tokens": 500,
    "stop_words": ["bye", "exit"],
    "debug": False,
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)          # indent makes it human-readable

with open("config.json") as f:
    loaded = json.load(f)

print(loaded)
print(type(loaded["temperature"]), type(loaded["debug"]))   # types survive, unlike CSV

{'model': 'gpt-4o-mini', 'temperature': 0.2, 'max_tokens': 500, 'stop_words': ['bye', 'exit'], 'debug': False}
<class 'float'> <class 'bool'>


In [9]:
# An LLM API reply is JSON text - json.loads turns it into a dict you can index
reply = '{"choices": [{"message": {"role": "assistant", "content": "Helllo!"}}]}'
data = json.loads(reply)
print(data["choices"][0]["message"]["content"])

Helllo!


In [10]:
# BREAKS ON PURPOSE - JSON is stricter than Python
bad = "{'model': 'gpt-4o-mini'}"       # single quotes are fine in Python, not in JSON
json.loads(bad)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

**JSONDecodeError: Expecting property name enclosed in double quotes.**

JSON has stricter rules than Python:
- Use double quotes `" "`, not single quotes `' '`.
- Write `true`, `false`, `null` in small letters.
- No extra comma after the last item.

Sometimes an API or an AI model sends broken JSON. Then you will see this error. So in real code, put `json.loads` inside `try` / `except json.JSONDecodeError`.

# 7. Real example: load the settings safely

An app reads its settings from a file when it starts. But what if the file is missing or broken? The app should not crash. It should use default settings.

This example uses everything from today: `try`, `except`, files and JSON.

In [ ]:
DEFAULTS = {"model": "gpt-4o-mini", "temperature": 0.7, "max_tokens": 256}

def load_config(path):
    try:
        with open(path) as f:
            user = json.load(f)
    except FileNotFoundError:
        print(f"{path} not found - using defaults")
        return DEFAULTS
    except json.JSONDecodeError as e:
        print(f"{path} is not valid JSON ({e}) - using defaults")
        return DEFAULTS
    return {**DEFAULTS, **user}             # file values override defaults

with open("broken.json", "w") as f:
    f.write("{'oops': True}")

print(load_config("config.json"))
print(load_config("missing.json"))
print(load_config("broken.json"))

# 8. Your turn

**1.** Make a function `safe_divide(a, b)`.
- If `b` is `0`, raise your own error `DivisionByZeroError`. (Make it a class that inherits from `Exception`.)
- If not, return `a / b`.
- Call it inside `try` / `except` with `b = 0`, and print a friendly message.
- Add `finally` that prints `"done"`.

**2.** Ask the user for their age with `input()`.
- Keep asking (use a `while` loop) until they type a correct whole number from 1 to 120.
- Catch `ValueError` when they type something that is not a number.
- If the number is below 1 or above 120, `raise` your own `ValueError`.

**3.** Make a list of 5 products. Each product has `name`, `price` and `stock`.
- Save the list to `products.csv`.
- Read the file back. Change `price` and `stock` to numbers.
- Print the total value of all stock: add up `price × stock` for every product.
- Add one row where `"price"` is `"N/A"`. Your code should skip this row and print a warning. It should not crash.

**4.** Make a JSON settings file for a chatbot with: `bot_name`, `greeting`, `max_turns`, and `banned_words` (a list).
- Load the file and print the greeting.
- Change `max_turns`, and save the file again.
- Load it one more time, and print `max_turns` to show the change was saved.